In [1]:
import torch
from txtai import Embeddings
import pandas as pd
#from datasets import load_dataset
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
import bm25s

/home/droon/miniconda3/envs/torchtest/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.14.1+cu126
True
NVIDIA GeForce RTX 4080 Laptop GPU


In [67]:
train = pd.read_json('pubmedqa/data/pqaa_train_set.json').T
dev = pd.read_json('pubmedqa/data/pqaa_dev_set.json').T
contexts_train = [" ".join(contexts) for contexts in train["CONTEXTS"]]
context_ids = train.index.astype(str).tolist()
documents = list(zip(context_ids, contexts_train))
print(train.head())

                                                   QUESTION  \
22117569  Is an advance care planning model feasible in ...   
16097998  Is coeliac disease screening in risk groups ju...   
15223725  Does blood pressure change in treated hyperten...   
17274051  Metastatic carcinoma to the cervical nodes fro...   
27581329  Estimation of basal metabolic rate in Chinese:...   

                                                   CONTEXTS  \
22117569  [An effective advance care planning programme ...   
16097998  [The benefits of serologic screening for coeli...   
15223725  [To determine whether there are differences be...   
17274051  [The aim of the study was to evaluate the outc...   
27581329  [Measurement of basal metabolic rate (BMR) is ...   

                                                     LABELS  \
22117569  [BACKGROUND, METHODS, SETTINGS AND PARTICIPANT...   
16097998                     [BACKGROUND, METHODS, RESULTS]   
15223725                      [OBJECTIVES, METHOD, RE

In [68]:
# dense embeddings
dense_embeddings = Embeddings(path="neuml/pubmedbert-base-embeddings", content=True)
dense_embeddings.index(documents)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2242.28it/s]


In [70]:
# Test dense embeddings search
test_dense = dense_embeddings.search("Is an advance care planning model feasible in community palliative care?")

[d['id'] for d in test_dense]

['22117569', '21592383', '18693227']

In [75]:
test_dense[0]

{'id': '22117569',
 'text': 'An effective advance care planning programme involves an organizational wide commitment and preparedness for health service reform to embed advance care planning into routine practice. Internationally, such programmes have been implemented predominantly in aged and acute care with more recent work in primary care. A multi-site action research was conducted over a 16-month period in 2007-2009 with three Victorian community palliative care services. Using mixed method data collection strategies to assess feasibility, we conducted a baseline audit of staff and clients; analysed relevant documents (client records, policies, procedures and quality improvement strategies) pre-implementation and post-implementation and conducted key informant interviews (n\u2003=\u20039). Three community palliative care services: one regional and two metropolitan services in Victoria, Australia. The services demonstrated that it was feasible to embed the Model into their organizat

<h2>Sparse Embeddings</h2>

In [71]:


contexts_train = [" ".join(contexts) for contexts in train["CONTEXTS"]]
context_ids = train.index.to_numpy()
contexts_train_tokens = bm25s.tokenize(contexts_train, stopwords="en")

sparse_embeddings = bm25s.BM25(corpus=context_ids)
sparse_embeddings.index(contexts_train_tokens)

assert sparse_embeddings.scores["num_docs"] == len(train)
print(f"Indexed {len(contexts_train):,} contexts with BM25.")

Split strings:   0%|          | 0/1000 [00:00<?, ?it/s]

Indexed 1,000 contexts with BM25.


<h2>RRF<h2>

In [72]:
sparse_ids, _ = sparse_embeddings.retrieve(query_tokens, k=5,return_as = 'tuple')
sparse_ids[0].tolist()

[22117569, 10158597, 21592383, 21658267, 11458136]

In [74]:
from typing import List, Dict

def reciprocal_rank_fusion(ranks: List[List[str]], k: int = 60) -> List[str]:
    """
    ranks: list of lists of document IDs from each retrieval method
    k: constant for rank discount
    """
    doc_scores = {doc: 0.0 for doc in set().union(*ranks)}
    for rank_list in ranks:
        for rank, doc in enumerate(rank_list, start=1):
            doc_scores[doc] += 1 / (rank + k)
    return sorted(doc_scores.keys(), key=lambda d: doc_scores[d], reverse=True)

# Example usage:
# list1 = ["docA", "docB", "docC"]
# list2 = ["docB", "docC", "docD"]
# final = reciprocal_rank_fusion([list1, list2], k=60)
# print(final)

def retrieve_and_fuse(query,dense_embeddings, sparse_embeddings):
    dense_list= dense_embeddings.search(query)
    dense_ids = [dict['id'] for dict in dense_list]
    query_tokens = bm25s.tokenize(query, stopwords="en")
    sparse_ids, _ = sparse_embeddings.retrieve(query_tokens, k=5)
    sparse_ids = sparse_ids[0].tolist()
    fused_list = reciprocal_rank_fusion([dense_ids, sparse_ids])
    return fused_list

fused_list = retrieve_and_fuse("Is an advance care planning model feasible in community palliative care?", dense_embeddings, sparse_embeddings)
fused_list = [int(id) for id in fused_list]
train.loc[fused_list]

,QUESTION,CONTEXTS,LABELS,MESHES,YEAR,reasoning_required_pred,reasoning_free_pred,final_decision,LONG_ANSWER
22117569,Is an advance care planning model feasible in ...,[An effective advance care planning programme ...,"[BACKGROUND, METHODS, SETTINGS AND PARTICIPANT...","[Adult, Advance Care Planning, Advance Directi...",2012,yes,yes,yes,An advance care planning model is feasible for...
22117569,Is an advance care planning model feasible in ...,[An effective advance care planning programme ...,"[BACKGROUND, METHODS, SETTINGS AND PARTICIPANT...","[Adult, Advance Care Planning, Advance Directi...",2012,yes,yes,yes,An advance care planning model is feasible for...
10158597,Does a dedicated discharge coordinator improve...,[To evaluate the effectiveness of the role of ...,"[OBJECTIVE, DESIGN, SETTING, PATIENTS, INTERVE...","[Activities of Daily Living, Health Status Ind...",1996,yes,yes,yes,The introduction of a discharge coordinator im...
21592383,Canada's Compassionate Care Benefit: is it an ...,[An increasingly significant public health iss...,"[BACKGROUND, METHODS, RESULTS]","[Aged, Canada, Caregivers, Cost of Illness, Fe...",2011,no,no,no,"This study, from the perspective of family car..."
18693227,Does a geriatric oncology consultation modify ...,[This study was performed to describe the trea...,"[BACKGROUND, PATIENTS AND METHODS, RESULTS]","[Activities of Daily Living, Affect, Aged, Age...",2008,yes,yes,yes,The geriatric oncology consultation led to a m...
21592383,Canada's Compassionate Care Benefit: is it an ...,[An increasingly significant public health iss...,"[BACKGROUND, METHODS, RESULTS]","[Aged, Canada, Caregivers, Cost of Illness, Fe...",2011,no,no,no,"This study, from the perspective of family car..."
21658267,"Do improvements in outreach, clinical, and fam...",[There are three main service delivery channel...,"[BACKGROUND, METHODS, RESULTS]","[Adolescent, Adult, Africa, Asia, Caribbean Re...",2011,yes,yes,yes,Our findings support the importance of increas...
11458136,Does managed care enable more low income perso...,[By requiring or encouraging enrollees to obta...,"[BACKGROUND, OBJECTIVES, RESEARCH DESIGN, SUBJ...","[Adult, Continuity of Patient Care, Cross-Sect...",2001,no,maybe,maybe,Having a usual physician can be an effective t...
